In [ ]:
# enable clicking by mouse in the Pyplot drawings
%pip install ipympl -q
%matplotlib widget

In [ ]:
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [ ]:
# select device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
# download variational autoencoder
!wget http://agentspace.org/download/mnist_cvae_069_encoder.pth
!wget http://agentspace.org/download/mnist_cvae_069_decoder.pth

In [ ]:
# load variational autoencoder
encoder = torch.load('mnist_cvae_069_encoder.pth', weights_only=False, map_location=device)
decoder = torch.load('mnist_cvae_069_decoder.pth', weights_only=False, map_location=device)

In [ ]:
encoder.eval()

In [ ]:
decoder.eval()

In [ ]:
# get datasets
train_dataset = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root='data', train=False, download=True, transform=transforms.ToTensor())

In [ ]:
# select a subset of classes
classes_to_keep = [0, 6, 9]
class_to_id = {class_: i for i, class_ in enumerate(classes_to_keep)}
print(class_to_id)
id_to_class = {i: class_ for class_, i in class_to_id.items()}
print(id_to_class)

In [ ]:
class RemappedSubset(Dataset):
    def __init__(self, original_dataset, indices, class_to_new):
        self.original = original_dataset
        self.indices = indices
        self.class_to_new = class_to_new

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        x, y = self.original[real_idx]
        new_y = self.class_to_new[int(y)]
        return x, new_y

In [ ]:
# create subset
train_indices = [i for i, (_, label) in enumerate(train_dataset) if label in classes_to_keep] # indices for train subset
test_indices = [i for i, (_, label) in enumerate(test_dataset) if label in classes_to_keep] # indices for test subset
train_subset = RemappedSubset(train_dataset, train_indices, class_to_id)
test_subset = RemappedSubset(test_dataset, test_indices, class_to_id)

In [ ]:
# test subset dataset
labels = torch.tensor([label for _, label in train_subset])
print(torch.unique(labels))
labels = torch.tensor([label for _, label in test_subset])
print(torch.unique(labels))

In [ ]:
# get few samples
x_samples = torch.stack([ test_subset[i][0] for i in [0,1,2,3,4,5,6,7,8,9] ])
print(x_samples.shape)

In [ ]:
# encode and decode
y_samples = decoder(encoder(x_samples.to(device)))

In [ ]:
# show the comparison
plt.close('all')
plt.figure(figsize=(20, 4))
for i in range(10):
    input_img = (x_samples[i].squeeze(0).detach().cpu().numpy()*255).astype(np.uint8)
    plt.subplot(2, 10, i+1)
    plt.imshow(input_img, cmap='gray')
    plt.axis('off')
    output_img = (y_samples[i].squeeze(0).detach().cpu().numpy()*255).astype(np.uint8)
    plt.subplot(2, 10, i+1+10)
    plt.imshow(output_img, cmap='gray')
    plt.axis('off')
plt.show()

In [ ]:
# Plot latent space
plt.close('all')
n = 20
norm = torch.distributions.Normal(0, 1)
grid_x = 3*norm.icdf(torch.linspace(0.05, 0.95, n-1))
grid_y = 3*norm.icdf(torch.linspace(0.05, 0.95, n-1))
figure = np.zeros((28 * (n-1), 28 * (n-1)))
with torch.no_grad():
    for i, yi in enumerate(grid_x):
        for j, xi in enumerate(grid_y):
            z = torch.tensor([[xi, yi]]).float().to('cuda')
            x_decoded = decoder(z)
            digit = x_decoded[0].reshape(28, 28).cpu().numpy()
            figure[i * 28: (i + 1) * 28, j * 28: (j + 1) * 28] = digit

plt.figure(figsize=(6, 6))
plt.imshow(figure, cmap='Greys_r')
plt.axis('off')
coords = []

def onclick(event):
    if event.xdata is not None and event.ydata is not None:
        x, y = event.xdata, event.ydata
        coords.append((x, y))
        print(f"Clicked at pixel coords: x={x:.2f}, y={y:.2f}")

cid = fig.canvas.mpl_connect('button_press_event', onclick)

In [ ]:
# representants
model = EmbedWipeout(28,28,3,3).to(device)

In [ ]:
# create train and test dataloader
batch_size = 512
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

In [ ]:
# autoencoder architecture
class Autoencoder(nn.Module):
    def __init__(self, classifier, h=28, w=28, d=3):
        super().__init__()
        self.encoder = nn.Sequential(
            classifier.embed,
            classifier.norm,
        )
        self.decoder = nn.Sequential(
            nn.Linear(d, 4*h*w),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(4*h*w, h*w),
            nn.Sigmoid(),
            nn.Unflatten(1, (1, h, w)),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

In [ ]:
autoencoder = Autoencoder(model,28,28,3).to(device)

In [ ]:
for param in autoencoder.encoder.parameters():
    param.requires_grad = False

In [ ]:
# Define optimizer
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
# scheduler
step_size = 16
gamma = 0.7
scheduler = StepLR(optimizer, step_size=step_size, gamma=gamma)

In [ ]:
# training
epoch = 0

In [ ]:
autoencoder.train()
num_epochs = 16
for _ in range(num_epochs):
    # to record loss and accuracy
    batch_loss = np.array([])
    batch_acc = np.array([])

    for x_train, _ in train_loader:

        # send data to device
        input = x_train.to(device)

        # reset parameters gradient to zero
        optimizer.zero_grad()

        # forward pass to the model
        output = autoencoder(input)

        # cross entropy loss
        loss = F.binary_cross_entropy(output, input)

        # find gradients
        loss.backward()

        # update parameters using gradients
        optimizer.step()

        # recording loss
        batch_loss = np.append(batch_loss, [loss.item()])

        # recording accuracy
        total_train = input.numel()
        correct_train = (torch.abs(input-output) < 0.1).sum().item()
        acc = (100.0 * correct_train) / total_train
        batch_acc = np.append(batch_acc, [acc])

    epoch_loss = batch_loss.mean()
    epoch_acc = batch_acc.mean()

    print(f'Epoch: {epoch} Loss: {epoch_loss:.6f} Acc: {epoch_acc:.4f} Learning Rate: {scheduler.get_last_lr()[0]:.7f}')
    scheduler.step()
    epoch += 1

In [ ]:
def train(num_epochs):
    global epoch
    for _ in range(num_epochs):
        # to record loss and accuracy
        batch_loss = np.array([])
        batch_acc = np.array([])

        for x_train, _ in train_loader:

            # send data to device
            input = x_train.to(device)

            # reset parameters gradient to zero
            optimizer.zero_grad()

            # forward pass to the model
            output = autoencoder(input)

            # cross entropy loss
            loss = F.binary_cross_entropy(output, input)

            # find gradients
            loss.backward()

            # update parameters using gradients
            optimizer.step()

            # recording loss
            batch_loss = np.append(batch_loss, [loss.item()])

            # recording accuracy
            total_train = input.numel()
            correct_train = (torch.abs(input-output) < 0.1).sum().item()
            acc = (100.0 * correct_train) / total_train
            batch_acc = np.append(batch_acc, [acc])

        epoch_loss = batch_loss.mean()
        epoch_acc = batch_acc.mean()

        print(f'Epoch: {epoch} Loss: {epoch_loss:.6f} Acc: {epoch_acc:.4f} Learning Rate: {scheduler.get_last_lr()[0]:.7f}')
        scheduler.step()
        epoch += 1

In [ ]:
# training
print(f"number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
num_epochs = 16
train(num_epochs)

In [ ]:
# test
sample = train_dataset[train_indices[0]]
print(sample[0].shape,sample[1])

In [ ]:
probabilities = model(sample[0].unsqueeze(0).to(device))
print(probabilities.argmax().item())

In [ ]:
sample_ = sample[0].squeeze(0).detach().numpy()
plt.imshow(sample_, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
sample_embedding = model.norm(model.embed(sample[0].to(device)))[0]
print(sample_embedding)
sample_unnormalized_embedding = model.embed(sample[0].to(device))[0]
print(sample_unnormalized_embedding)

In [ ]:
# pseudo-inverse
recovered_unnormalized_embeding = (sample_embedding - model.norm.bias) / model.norm.weight * torch.sqrt(var + model.norm.eps) + mean
print(recovered_unnormalized_embeding)

In [ ]:
(recovered_unnormalized_embeding - model.embed[1].bias).shape

In [ ]:
torch.linalg.pinv(model.embed[1].weight).shape

In [ ]:
recovered_sample = ((recovered_unnormalized_embeding - model.embed[1].bias) @ torch.linalg.pinv(model.embed[1].weight).t()).reshape(28,28)
print(recovered_sample.shape, recovered_sample.min(), recovered_sample.max())

In [ ]:
recovered_sample_ = recovered_sample.squeeze(0).detach().cpu().numpy()
plt.imshow(recovered_sample_, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
# representatives
vecs = model.wipeout.weight.detach().cpu().numpy()
print(vecs.shape)
print(vecs)

In [ ]:
# visualize representatives
plt.close('all')
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# colormap with one distinct color per vector
colors = plt.cm.tab10(range(len(vecs)))

for i, v in enumerate(vecs):
    x, y, z = v.tolist()
    ax.quiver(0, 0, 0, x, y, z, color=colors[i], label=f"{id_to_class[i]}")
    ax.scatter([x], [y], [z], color=colors[i])

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")

# place legend outside plot for readability
ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.show()


In [ ]:
sample = train_dataset[train_indices[1]]

In [ ]:
embedding = model.norm(model.embed(sample[0].to(device)))[0].detach().cpu().numpy()
print(embedding.shape)
print(embedding)

In [ ]:
embedding @ vecs.T

In [ ]:
id_to_class[np.argmax(embedding @ vecs.T)]

In [ ]:
sample_ = sample[0].squeeze(0).detach().numpy()
plt.close('all')
plt.imshow(sample_, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
# calculate embeddings norms
norms = torch.norm(embeddings, dim=1)
norms = norms.detach().cpu().numpy()
print(norms.shape, norms.min(), norms.max(), norms.mean())

In [ ]:
# draw the norms distribution
plt.close('all')
plt.hist(norms, bins=20)
plt.xlabel('Norm')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# visualize latent space before normalization
plt.close('all')
points = unnormalized_embedings.detach().cpu().numpy()
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

for i in range(3):  # assuming 3 categories: 0,1,2
    mask = categories == i
    pts = points[mask]
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], color=colors[i], label=id_to_class[i])

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")

# place legend outside plot for readability
ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.show()

In [ ]:
# visualize latent space
plt.close('all')
points = embeddings.detach().cpu().numpy()
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

for i in range(3):  # assuming 3 categories: 0,1,2
    mask = categories == i
    pts = points[mask]
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], color=colors[i], label=id_to_class[i])

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")

# place legend outside plot for readability
ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.show()